# Train YOLO một lớp trên Google Colab

Notebook này huấn luyện YOLO để tìm một lớp duy nhất `object`. YOLO chỉ định vị vùng hộp, vỉ hoặc viên thuốc; MobileNet xử lý phân loại crop ở bước sau.

Notebook sẽ:

1. Kiểm tra GPU và tải dataset từ Roboflow.
2. Kiểm tra số ảnh, nhãn và ảnh thiếu label.
3. Huấn luyện YOLO với checkpoint tốt nhất được lưu vào Google Drive.
4. Báo cáo mAP@50, mAP@75, mAP@[50:95], thời gian train, tốc độ suy luận và số tham số.
5. Đánh giá nhiều confidence threshold để giảm false positive trên background.

**Thiết lập trước khi chạy:** bật `Runtime > Change runtime type > T4 GPU`, tạo Colab Secret tên `ROBOFLOW_API_KEY`, và mount Google Drive khi được hỏi.

In [ ]:
!pip install -q ultralytics roboflow pyyaml

In [ ]:
import json
import os
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
import ultralytics
import yaml
from google.colab import userdata
from roboflow import Roboflow
from ultralytics import YOLO

print('torch      :', torch.__version__)
print('ultralytics:', ultralytics.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU: Runtime > Change runtime type > T4 GPU')
print('GPU        :', torch.cuda.get_device_name(0))

RF_API_KEY = userdata.get('ROBOFLOW_API_KEY')
if not RF_API_KEY:
    raise RuntimeError('Hãy tạo Colab Secret ROBOFLOW_API_KEY trước khi chạy.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace('meo-meo-iopfm').project('final2-m8rcv')
version = project.version(5)
dataset = version.download('yolov8')

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / 'data.yaml'
with DATA_YAML.open('r', encoding='utf-8') as file:
    data_cfg = yaml.safe_load(file)

names = data_cfg['names']
if isinstance(names, dict):
    names = [names[key] for key in sorted(names)]
if names != ['object']:
    raise ValueError(f'Notebook yêu cầu đúng một lớp object, nhận được: {names}')

def split_image_dir(split):
    value = data_cfg.get(split)
    if not value:
        return None
    path = Path(value)
    return path if path.is_absolute() else DATASET_DIR / path

print('Dataset:', DATASET_DIR)
print('YAML   :', DATA_YAML)

In [ ]:
def image_files(directory):
    if directory is None or not directory.is_dir():
        return []
    return [path for path in directory.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}]

def check_split(split):
    image_dir = split_image_dir(split)
    if image_dir is None:
        return {'split': split, 'images': 0, 'labels': 0, 'missing_labels': 0, 'instances': 0}
    label_dir = image_dir.parent / 'labels'
    images = image_files(image_dir)
    labels = {path.stem for path in label_dir.glob('*.txt')} if label_dir.is_dir() else set()
    missing = [path for path in images if path.stem not in labels]
    instances = 0
    invalid_lines = 0
    for label_path in label_dir.glob('*.txt') if label_dir.is_dir() else []:
        for line in label_path.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            fields = line.split()
            instances += 1
            valid = len(fields) == 5 and fields[0] == '0'
            if valid:
                try:
                    valid = all(0 <= float(value) <= 1 for value in fields[1:])
                except ValueError:
                    valid = False
            invalid_lines += int(not valid)
    
    return {
        'split': split,
        'images': len(images),
        'labels': len(labels),
        'missing_labels': len(missing),
        'instances': instances,
        'invalid_label_lines': invalid_lines,
    }

split_table = pd.DataFrame([check_split(split) for split in ['train', 'valid', 'val', 'test']])
split_table = split_table[split_table['images'] > 0]
display(split_table)

if split_table['missing_labels'].any() or split_table['invalid_label_lines'].any():
    raise ValueError('Dataset còn ảnh thiếu label hoặc label sai format. Sửa dataset trước khi train.')

EVAL_SPLIT = 'test' if 'test' in split_table['split'].values else 'valid'
print('Split đánh giá:', EVAL_SPLIT)

In [ ]:
from google.colab import drive

MODEL_NAME = 'yolov9s.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
PATIENCE = 20
SEED = 0
PROJECT_DIR = '/content/drive/MyDrive/KetQua_NCKH'
RUN_NAME = f'yolo_object_seed{SEED}'

model = YOLO(MODEL_NAME)
parameter_count = sum(parameter.numel() for parameter in model.model.parameters())
train_start = time.perf_counter()
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    seed=SEED,
    deterministic=True,
    patience=PATIENCE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=False,
    plots=True,
    save=True,
    val=True,
    verbose=True,
)
train_minutes = (time.perf_counter() - train_start) / 60
best_path = Path(model.trainer.best)
print('Best checkpoint:', best_path)
print('Training minutes:', round(train_minutes, 2))
print('Parameters:', parameter_count, f'({parameter_count / 1e6:.3f}M)')

In [ ]:
best_model = YOLO(str(best_path))
metrics = best_model.val(
    data=str(DATA_YAML),
    split=EVAL_SPLIT,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    plots=True,
    verbose=True,
)

inference_ms = float(metrics.speed.get('inference', 0.0))
report = {
    'model': MODEL_NAME,
    'split': EVAL_SPLIT,
    'mAP50': float(metrics.box.map50),
    'mAP75': float(metrics.box.map75),
    'mAP50-95': float(metrics.box.map),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'training_minutes': round(train_minutes, 3),
    'inference_ms_per_image': round(inference_ms, 3),
    'inference_images_per_second': round(1000 / inference_ms, 3) if inference_ms else None,
    'parameters': parameter_count,
    'parameters_million': round(parameter_count / 1e6, 3),
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'patience': PATIENCE,
    'seed': SEED,
}
print(json.dumps(report, indent=2))
Path(model.trainer.save_dir, 'report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

In [ ]:
threshold_rows = []
for confidence in [0.25, 0.40, 0.50, 0.60, 0.70]:
    threshold_metrics = best_model.val(
        data=str(DATA_YAML),
        split=EVAL_SPLIT,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        workers=2,
        conf=confidence,
        plots=False,
        verbose=False,
    )
    threshold_rows.append({
        'confidence': confidence,
        'precision': float(threshold_metrics.box.mp),
        'recall': float(threshold_metrics.box.mr),
        'mAP50': float(threshold_metrics.box.map50),
        'mAP50-95': float(threshold_metrics.box.map),
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)
threshold_table.to_csv(Path(model.trainer.save_dir, 'confidence_sweep.csv'), index=False)
print('Chọn confidence có precision cao nhưng recall vẫn phù hợp với mục tiêu crop.')

## Cách đọc kết quả

- `object -> object` lớn: tìm đúng vật thể.
- `background -> object` lớn: khoanh nhầm nền, cần tăng confidence hoặc thêm ảnh nền khó.
- `object -> background` lớn: bỏ sót vật thể, cần sửa nhãn hoặc tăng dữ liệu.
- `mAP@50` đánh giá dễ hơn; `mAP@[50:95]` khắt khe hơn vì yêu cầu box khớp chặt hơn.

Các file dùng cho thuyết minh nằm trong thư mục run: `report.json`, `confidence_sweep.csv`, `results.csv`, `confusion_matrix.png`, `confusion_matrix_normalized.png`, `results.png`, các đường cong Precision/Recall/F1 và `weights/best.pt`.